# Notebook 03 — Persistent 3D Patch Extraction for RibFrac

This notebook extracts fixed-size 3D CT patches and multi-task targets from the persistent preprocessed RibFrac dataset produced by Notebook 02.

## Dataset
- Labelled cohort: 500 patients
- Training Part 1: 300
- Training Part 2: 120
- Labelled validation: 80
- Patient-level split: 350 train / 75 validation / 75 internal held-out test
- Split seed: 42

## Patch policy
- Patch size: `(64, 64, 64)`
- Stride: `(32, 32, 32)`
- Segmentation target: `0 = background`, `1 = valid fracture`
- Classification target: `0 = background`, `1 = displaced`, `2 = non-displaced`, `3 = buckle`, `4 = segmental`
- `label_code = -1` is ambiguous and is excluded.
- Positive patches contain at least one valid fracture voxel.
- Negative patches contain no valid fracture voxels.
- Negative sampling is capped at a 1:1 negative-to-positive ratio.
- Mixed-class ties are excluded because classification uses a single-label softmax target.
- No patch-level train/validation/test split is performed.

**Important:** Notebook 03 does not contact Zenodo and does not redo CT preprocessing. It consumes the `.npy` volumes generated by Notebook 02.

In [1]:
# Cell 1 — Imports and reproducibility

import gc
import json
import random
import time
from pathlib import Path
from collections import Counter
import shutil

import numpy as np
import pandas as pd


import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")
print("Random seed:", SEED)

Libraries imported successfully.
Random seed: 42


In [2]:
# Cell 2 — Kaggle input and persistent patch-output configuration

# Published Notebook 02 dataset.
# Change only DATASET_ROOT if Kaggle displays a different mount path.
DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

PREPROCESSED_ROOT = DATASET_ROOT
IMAGE_DIR = PREPROCESSED_ROOT / "images"
LABEL_DIR = PREPROCESSED_ROOT / "labels"
METADATA_DIR = PREPROCESSED_ROOT / "metadata"

PATCH_ROOT = Path(
    "/kaggle/working/ribfrac-patch-dataset-final"
)
PATCH_METADATA_DIR = PATCH_ROOT / "metadata"
PATCH_STATUS_PATH = PATCH_METADATA_DIR / "processing_status.csv"

for path in [PATCH_ROOT, PATCH_METADATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Input dataset:", DATASET_ROOT)
print("Image directory:", IMAGE_DIR)
print("Label directory:", LABEL_DIR)
print("Metadata directory:", METADATA_DIR)
print("Patch output:", PATCH_ROOT)

Input dataset: /kaggle/input/datasets/waboke/ribfrac-preprocessed-final
Image directory: /kaggle/input/datasets/waboke/ribfrac-preprocessed-final/images
Label directory: /kaggle/input/datasets/waboke/ribfrac-preprocessed-final/labels
Metadata directory: /kaggle/input/datasets/waboke/ribfrac-preprocessed-final/metadata
Patch output: /kaggle/working/ribfrac-patch-dataset-final


In [3]:
print("Input dataset exists:", DATASET_ROOT.exists())
print("Images directory exists:", IMAGE_DIR.exists())
print("Labels directory exists:", LABEL_DIR.exists())
print("Metadata directory exists:", METADATA_DIR.exists())

print("Available CT files:",
      len(list(IMAGE_DIR.glob("*-image.npy"))))

print("Available label files:",
      len(list(LABEL_DIR.glob("*-label.npy"))))

Input dataset exists: True
Images directory exists: True
Labels directory exists: True
Metadata directory exists: True
Available CT files: 49
Available label files: 49


In [4]:
# Cell 3 — Patch and target configuration

PATCH_SIZE = (64, 64, 64)
STRIDE = (32, 32, 32)

NEGATIVE_TO_POSITIVE_RATIO = 1.0
MIN_FRACTURE_VOXELS = 1
AMBIGUOUS_LABEL_CODE = -1

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

LABEL_CODE_TO_CLASS = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
}

EXPECTED_SPLIT_COUNTS = {
    "train": 350,
    "validation": 75,
    "test": 75,
}

MAX_PATIENTS_PER_RUN = 5
RESUME_EXISTING = True

print("Patch size:", PATCH_SIZE)
print("Stride:", STRIDE)
print("Negative/positive ratio:", NEGATIVE_TO_POSITIVE_RATIO)
print("Classification classes:", CLASS_NAMES)
print("MAX_PATIENTS_PER_RUN:", MAX_PATIENTS_PER_RUN)

Patch size: (64, 64, 64)
Stride: (32, 32, 32)
Negative/positive ratio: 1.0
Classification classes: {0: 'background', 1: 'displaced', 2: 'non-displaced', 3: 'buckle', 4: 'segmental'}
MAX_PATIENTS_PER_RUN: 5


## Input validation

Notebook 02 stores one CT and one label volume per patient:

```text
images/RibFracXXX-image.npy
labels/RibFracXXX-label.npy
metadata/
    patient_source_registry_500.csv
    patient_split_500.csv
    combined_ribfrac_metadata.csv
    processing_status.csv
```

The patch extractor uses the existing patient registry and split. It does not recreate the split.

In [5]:
# Cell 4 — Verify persistent Notebook 02 dataset

for path in [IMAGE_DIR, LABEL_DIR, METADATA_DIR]:
    assert path.exists(), f"Required directory not found: {path}"

SPLIT_PATH = METADATA_DIR / "patient_split_500.csv"
REGISTRY_PATH = METADATA_DIR / "patient_source_registry_500.csv"
PROCESSING_STATUS_PATH = METADATA_DIR / "processing_status.csv"

assert SPLIT_PATH.exists(), f"Missing patient split: {SPLIT_PATH}"
assert REGISTRY_PATH.exists(), f"Missing patient registry: {REGISTRY_PATH}"

patient_split = pd.read_csv(SPLIT_PATH)
patient_registry = pd.read_csv(REGISTRY_PATH)

print("Persistent dataset verification: PASS")
print("Patients in split:", len(patient_split))
print("Patients in registry:", len(patient_registry))
print("\nSplit counts:")
print(patient_split["split"].value_counts())

Persistent dataset verification: PASS
Patients in split: 500
Patients in registry: 500

Split counts:
split
train         350
validation     75
test           75
Name: count, dtype: int64


In [6]:

# ============================================================
# CELL 4A — VERIFY AVAILABLE PREPROCESSED PATIENTS
# ============================================================

image_ids = {
    p.name.removesuffix("-image.npy")
    for p in IMAGE_DIR.glob("*-image.npy")
}

label_ids = {
    p.name.removesuffix("-label.npy")
    for p in LABEL_DIR.glob("*-label.npy")
}

complete_ids = image_ids & label_ids

print("=" * 60)
print("RIBFRAC PREPROCESSED DATASET AUDIT")
print("=" * 60)

print("CT volumes:", len(image_ids))
print("Fracture masks:", len(label_ids))
print("Complete patient pairs:", len(complete_ids))

assert image_ids == label_ids, (
    "Some CT volumes or fracture masks are missing."
)

assert len(complete_ids) == 49, (
    f"Expected 49 patients, found {len(complete_ids)}. "
    "Verify that the latest Kaggle checkpoint is attached."
)

available_patients = patient_split[
    patient_split["public_id"].astype(str).isin(complete_ids)
].copy()

assert len(available_patients) == 49

print("\nAvailable patients by split:")
print(available_patients["split"].value_counts())

print("\nAVAILABLE PATIENT DATASET: PASS")
print("=" * 60)


RIBFRAC PREPROCESSED DATASET AUDIT
CT volumes: 49
Fracture masks: 49
Complete patient pairs: 49

Available patients by split:
split
train    36
test     13
Name: count, dtype: int64

AVAILABLE PATIENT DATASET: PASS


In [7]:
# Cell 5 — Verify 500-patient split

def find_column(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

split_patient_col = find_column(patient_split, ["public_id", "patient_id", "id"])
registry_patient_col = find_column(patient_registry, ["public_id", "patient_id", "id"])

assert split_patient_col is not None, patient_split.columns.tolist()
assert registry_patient_col is not None, patient_registry.columns.tolist()

if split_patient_col != "public_id":
    patient_split = patient_split.rename(columns={split_patient_col: "public_id"})
if registry_patient_col != "public_id":
    patient_registry = patient_registry.rename(columns={registry_patient_col: "public_id"})

patient_split["public_id"] = patient_split["public_id"].astype(str)
patient_registry["public_id"] = patient_registry["public_id"].astype(str)

assert len(patient_split) == 500
assert patient_split["public_id"].nunique() == 500
assert set(patient_split["split"].unique()) == {"train", "validation", "test"}

actual_counts = patient_split["split"].value_counts().to_dict()
assert actual_counts == EXPECTED_SPLIT_COUNTS, actual_counts
assert set(patient_split["public_id"]) == set(patient_registry["public_id"])

train_patients = sorted(patient_split.loc[patient_split["split"]=="train", "public_id"])
validation_patients = sorted(patient_split.loc[patient_split["split"]=="validation", "public_id"])
test_patients = sorted(patient_split.loc[patient_split["split"]=="test", "public_id"])

print("Patient-level split verification: PASS")
print("Train:", len(train_patients))
print("Validation:", len(validation_patients))
print("Internal test:", len(test_patients))

Patient-level split verification: PASS
Train: 350
Validation: 75
Internal test: 75


In [8]:
# Cell 6 — Verify Notebook 02 processing status

if PROCESSING_STATUS_PATH.exists():
    processing_status = pd.read_csv(PROCESSING_STATUS_PATH)
    processing_status["public_id"] = processing_status["public_id"].astype(str)
    completed_ids = set(
        processing_status.loc[
            processing_status["status"] == "completed",
            "public_id"
        ]
    )
    print("Notebook 02 completed patients:", len(completed_ids))
    print(processing_status["status"].value_counts())
else:
    processing_status = pd.DataFrame()
    completed_ids = set()
    print("Warning: processing_status.csv was not found.")

Notebook 02 completed patients: 49
status
pending      451
completed     49
Name: count, dtype: int64


In [9]:

# Cell 6A — Verify completed CT-label pairs

image_ids = {
    p.name.removesuffix("-image.npy")
    for p in IMAGE_DIR.glob("*-image.npy")
}

label_ids = {
    p.name.removesuffix("-label.npy")
    for p in LABEL_DIR.glob("*-label.npy")
}

available_ids = image_ids & label_ids

print("=" * 60)
print("NOTEBOOK 02 — DATA AVAILABILITY CHECK")
print("=" * 60)

print("Patients marked completed:", len(completed_ids))
print("Available CT images:", len(image_ids))
print("Available fracture masks:", len(label_ids))
print("Complete CT-label pairs:", len(available_ids))

assert image_ids == label_ids, (
    "Some CT volumes or corresponding labels are missing."
)

assert completed_ids == available_ids, (
    "Processing status and saved CT-label pairs do not match."
)

assert len(available_ids) == 49, (
    "Expected 49 patient pairs. "
    "Check that the latest Kaggle checkpoint is attached."
)

print("\n49-PATIENT CHECKPOINT VERIFICATION: PASS")


NOTEBOOK 02 — DATA AVAILABILITY CHECK
Patients marked completed: 49
Available CT images: 49
Available fracture masks: 49
Complete CT-label pairs: 49

49-PATIENT CHECKPOINT VERIFICATION: PASS


## Annotation mapping

RibFrac label volumes contain annotation/instance IDs, not the final fracture classes.

The `label_code` in the patient metadata determines the fracture class:

- `-1` → ambiguous / ignore
- `1` → displaced
- `2` → non-displaced
- `3` → buckle
- `4` → segmental

Background is the voxel value `0` in the annotation volume.

Therefore, annotation ID `0` must not be confused with metadata `label_id = 0`.

In [10]:
# Cell 7 — Load combined annotation metadata

COMBINED_METADATA_PATH = METADATA_DIR / "combined_ribfrac_metadata.csv"
assert COMBINED_METADATA_PATH.exists(), f"Missing: {COMBINED_METADATA_PATH}"

combined_metadata = pd.read_csv(COMBINED_METADATA_PATH)

def first_existing_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None

patient_col = first_existing_column(combined_metadata, ["public_id", "patient_id", "id"])
label_id_col = first_existing_column(combined_metadata, ["label_id"])
label_code_col = first_existing_column(combined_metadata, ["label_code"])

assert patient_col is not None
assert label_id_col is not None
assert label_code_col is not None

combined_metadata = combined_metadata.rename(columns={
    patient_col: "public_id",
    label_id_col: "label_id",
    label_code_col: "label_code",
})

combined_metadata["public_id"] = combined_metadata["public_id"].astype(str)
combined_metadata["label_id"] = pd.to_numeric(combined_metadata["label_id"], errors="coerce")
combined_metadata["label_code"] = pd.to_numeric(combined_metadata["label_code"], errors="coerce")

print("Annotation metadata records:", len(combined_metadata))
print("Metadata mapping loaded successfully.")

Annotation metadata records: 4922
Metadata mapping loaded successfully.


In [11]:
# Cell 8 — Build per-patient annotation-ID mappings

patient_annotation_maps = {}

for patient_id, group in combined_metadata.groupby("public_id"):
    mapping = {}

    for _, row in group.iterrows():
        if pd.isna(row["label_id"]):
            continue

        annotation_id = int(row["label_id"])
        class_code = (
            AMBIGUOUS_LABEL_CODE
            if pd.isna(row["label_code"])
            else int(row["label_code"])
        )
        mapping[annotation_id] = class_code

    mapping[0] = 0
    patient_annotation_maps[str(patient_id)] = mapping

print("Patients with annotation metadata:", len(patient_annotation_maps))

example_patient = sorted(patient_annotation_maps.keys())[0]
print("Example patient:", example_patient)
print("Example mapping:", patient_annotation_maps[example_patient])

Patients with annotation metadata: 500
Example patient: RibFrac1
Example mapping: {0: 0, 1: 2, 2: 2}


In [13]:
# Cell 9 — Persistent patch storage helpers

def patient_patch_dir(split_name, patient_id):
    return PATCH_ROOT / split_name / str(patient_id)

def patient_status_path(split_name, patient_id):
    return patient_patch_dir(split_name, patient_id) / "patient_patch_status.json"

def patient_complete(split_name, patient_id):
    status_path = patient_status_path(split_name, patient_id)

    if not status_path.exists():
        return False

    try:
        with open(status_path, "r", encoding="utf-8") as f:
            status = json.load(f)
        return status.get("status") == "completed"
    except Exception:
        return False

print("Patch storage helpers ready.")

Patch storage helpers ready.


## Patch extraction logic

For each patient:

1. Load the persistent 1-mm CT and label arrays.
2. Generate `(64,64,64)` windows using stride `(32,32,32)`.
3. Convert annotation IDs to fracture classes using patient metadata.
4. Exclude patches containing ambiguous annotations.
5. Create a binary segmentation mask.
6. Assign a single five-class classification label to positive patches.
7. Exclude ties between multiple valid fracture classes.
8. Sample negative patches at a maximum 1:1 ratio.
9. Save the patient's patches and manifest.
10. Mark the patient as completed.

In [14]:
# Cell 10 — Generate patch start coordinates

def generate_patch_starts(shape, patch_size, stride):
    starts = []

    for d in range(0, shape[0] - patch_size[0] + 1, stride[0]):
        for h in range(0, shape[1] - patch_size[1] + 1, stride[1]):
            for w in range(0, shape[2] - patch_size[2] + 1, stride[2]):
                starts.append((d, h, w))

    return starts

test_starts = generate_patch_starts((128, 128, 128), PATCH_SIZE, STRIDE)
print("128³ test volume patch count:", len(test_starts))
assert len(test_starts) > 0

128³ test volume patch count: 27


In [16]:
# Cell 11 — Convert annotation IDs and generate targets

def build_class_volume(label_volume, annotation_mapping):
    class_volume = np.zeros(label_volume.shape, dtype=np.int8)

    for annotation_id in np.unique(label_volume):
        annotation_id = int(annotation_id)

        if annotation_id == 0:
            continue

        class_code = annotation_mapping.get(
            annotation_id,
            AMBIGUOUS_LABEL_CODE
        )

        class_volume[label_volume == annotation_id] = class_code

    return class_volume


def make_targets_from_class_patch(class_patch):
    ambiguous_mask = class_patch == AMBIGUOUS_LABEL_CODE

    if np.any(ambiguous_mask):
        return None, None, False, "ambiguous"

    valid_fracture = np.isin(class_patch, [1, 2, 3, 4])

    if not np.any(valid_fracture):
        segmentation = np.zeros((*PATCH_SIZE, 1), dtype=np.uint8)
        return segmentation, 0, False, "negative"

    segmentation = valid_fracture.astype(np.uint8)[..., None]

    class_counts = Counter(class_patch[valid_fracture].tolist())
    max_count = max(class_counts.values())

    dominant_classes = [
        cls for cls, count in class_counts.items()
        if count == max_count
    ]

    if len(dominant_classes) != 1:
        return None, None, False, "mixed_class_tie"

    return (
        segmentation,
        int(dominant_classes[0]),
        True,
        "positive",
    )

In [17]:

# Cell 11A — Verify target generation

test_label = np.zeros(PATCH_SIZE, dtype=np.int16)

# Simulate a non-displaced fracture instance
test_label[20:30, 20:30, 20:30] = 1

test_mapping = {
    0: 0,
    1: 2
}

test_class_volume = build_class_volume(
    test_label,
    test_mapping
)

seg_target, cls_target, positive, reason = (
    make_targets_from_class_patch(test_class_volume)
)

print("Segmentation shape:", seg_target.shape)
print("Classification target:", cls_target)
print("Positive patch:", positive)
print("Patch category:", reason)
print("Fracture voxels:", int(seg_target.sum()))

assert seg_target.shape == (64, 64, 64, 1)
assert cls_target == 2
assert positive is True
assert reason == "positive"
assert int(seg_target.sum()) == 1000

print("\nCell #11 target generation: PASS")


Segmentation shape: (64, 64, 64, 1)
Classification target: 2
Positive patch: True
Patch category: positive
Fracture voxels: 1000

Cell #11 target generation: PASS


In [18]:

# ============================================================
# CELL #12 — EXTRACT AND SAVE ONE PATIENT
# Memory-efficient 3D patch extraction
# ============================================================

def process_patient(patient_id, split_name):

    patient_id = str(patient_id)
    split_name = str(split_name)

    # --------------------------------------------------------
    # 1. Verify authoritative patient assignment
    # --------------------------------------------------------

    match = patient_split.loc[
        patient_split["public_id"].astype(str) == patient_id
    ]

    if len(match) != 1:
        raise ValueError(
            f"Patient missing from authoritative split: {patient_id}"
        )

    if str(match.iloc[0]["split"]) != split_name:
        raise ValueError(
            f"Incorrect split assignment for {patient_id}"
        )

    # --------------------------------------------------------
    # 2. Define input and output paths
    # --------------------------------------------------------

    image_path = IMAGE_DIR / f"{patient_id}-image.npy"
    label_path = LABEL_DIR / f"{patient_id}-label.npy"

    output_dir = patient_patch_dir(split_name, patient_id)
    status_path = patient_status_path(split_name, patient_id)

    if not image_path.is_file():
        raise FileNotFoundError(f"CT not found: {image_path}")

    if not label_path.is_file():
        raise FileNotFoundError(f"Label not found: {label_path}")

    if patient_id not in patient_annotation_maps:
        raise KeyError(
            f"No annotation metadata for {patient_id}"
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    required_outputs = [
        output_dir / "images.npy",
        output_dir / "segmentation.npy",
        output_dir / "classification.npy",
        output_dir / "patch_manifest.csv",
    ]

    # --------------------------------------------------------
    # 3. Skip verified existing patient output
    # --------------------------------------------------------

    if RESUME_EXISTING and patient_complete(
        split_name, patient_id
    ):

        try:
            if all(p.is_file() for p in required_outputs):

                with open(status_path, "r") as f:
                    existing_status = json.load(f)

                existing_arrays = [
                    np.load(p, mmap_mode="r", allow_pickle=False)
                    for p in required_outputs[:3]
                ]

                n = len(existing_arrays[0])

                if (
                    n > 0
                    and all(len(a) == n for a in existing_arrays)
                    and n == existing_status["saved_patches"]
                ):
                    del existing_arrays
                    print(f"{patient_id}: already extracted — SKIP")
                    return existing_status

        except (OSError, ValueError, KeyError):
            print("Existing output incomplete; reprocessing.")

    # --------------------------------------------------------
    # 4. Memory-mapped input volumes
    # --------------------------------------------------------

    ct = np.load(
        image_path,
        mmap_mode="r",
        allow_pickle=False
    )

    label = np.load(
        label_path,
        mmap_mode="r",
        allow_pickle=False
    )

    assert ct.ndim == 3
    assert label.ndim == 3
    assert ct.shape == label.shape
    assert ct.dtype == np.float32
    assert label.dtype == np.int16

    print("=" * 65)
    print("PATIENT PATCH EXTRACTION")
    print("=" * 65)

    print("Patient:", patient_id)
    print("Split:", split_name)
    print("Volume shape:", ct.shape)

    # --------------------------------------------------------
    # 5. Convert instance IDs into classification codes
    # --------------------------------------------------------

    class_volume = build_class_volume(
        label,
        patient_annotation_maps[patient_id]
    )

    # --------------------------------------------------------
    # 6. Generate candidate coordinates
    # --------------------------------------------------------

    starts = generate_patch_starts(
        ct.shape,
        PATCH_SIZE,
        STRIDE
    )

    print("Candidate patches:", len(starts))

    positive_records = []
    negative_starts = []

    skipped_ambiguous = 0
    skipped_ties = 0
    skipped_small = 0

    # --------------------------------------------------------
    # 7. First pass: classify candidate patches
    # No CT patch arrays accumulated in RAM
    # --------------------------------------------------------

    for d, h, w in starts:

        class_patch = class_volume[
            d:d+PATCH_SIZE[0],
            h:h+PATCH_SIZE[1],
            w:w+PATCH_SIZE[2]
        ]

        segmentation, classification, positive, reason = (
            make_targets_from_class_patch(class_patch)
        )

        if reason == "ambiguous":
            skipped_ambiguous += 1
            continue

        if reason == "mixed_class_tie":
            skipped_ties += 1
            continue

        if positive:

            if int(segmentation.sum()) < MIN_FRACTURE_VOXELS:
                skipped_small += 1
                continue

            positive_records.append(
                ((d, h, w), int(classification))
            )

        else:
            negative_starts.append((d, h, w))

    # --------------------------------------------------------
    # 8. Deterministic negative sampling
    # --------------------------------------------------------

    rng = np.random.default_rng(
        SEED + sum(ord(c) for c in patient_id)
    )

    n_negative = min(
        len(negative_starts),
        int(
            len(positive_records)
            * NEGATIVE_TO_POSITIVE_RATIO
        )
    )

    if n_negative:
        selected = rng.choice(
            len(negative_starts),
            size=n_negative,
            replace=False
        )

        selected_negatives = [
            negative_starts[i] for i in selected
        ]
    else:
        selected_negatives = []

    records = [
        (start, cls, 1)
        for start, cls in positive_records
    ]

    records.extend(
        (start, 0, 0)
        for start in selected_negatives
    )

    n_patches = len(records)

    if n_patches == 0:
        raise RuntimeError(
            f"No usable patches generated for {patient_id}"
        )

    print("Positive patches:", len(positive_records))
    print("Selected negative patches:", n_negative)
    print("Total patches to save:", n_patches)
    print("Skipped ambiguous:", skipped_ambiguous)
    print("Skipped mixed-class ties:", skipped_ties)

    # --------------------------------------------------------
    # 9. Check available storage
    # --------------------------------------------------------

    patch_voxels = int(np.prod(PATCH_SIZE))

    estimated_bytes = (
        n_patches * patch_voxels * 5
        + n_patches
    )

    free_bytes = shutil.disk_usage(output_dir).free

    if free_bytes < estimated_bytes + 512 * 1024**2:
        raise RuntimeError(
            "Insufficient disk space for patch extraction."
        )

    # --------------------------------------------------------
    # 10. Save patches using on-disk NumPy arrays
    # --------------------------------------------------------

    staging_dir = output_dir / ".incomplete"
    if staging_dir.exists():
        shutil.rmtree(staging_dir)

    staging_dir.mkdir(parents=True)

    image_shape = (n_patches, *PATCH_SIZE, 1)
    label_shape = (n_patches, *PATCH_SIZE, 1)

    images_mm = None
    masks_mm = None
    classes_mm = None

    try:

        images_mm = np.lib.format.open_memmap(
            staging_dir / "images.npy",
            mode="w+",
            dtype=np.float32,
            shape=image_shape
        )

        masks_mm = np.lib.format.open_memmap(
            staging_dir / "segmentation.npy",
            mode="w+",
            dtype=np.uint8,
            shape=label_shape
        )

        classes_mm = np.lib.format.open_memmap(
            staging_dir / "classification.npy",
            mode="w+",
            dtype=np.int8,
            shape=(n_patches,)
        )

        manifest_rows = []

        # ----------------------------------------------------
        # 11. Second pass: load and write selected patches
        # ----------------------------------------------------

        for index, (start, cls, is_positive) in enumerate(records):

            d, h, w = start

            ct_patch = ct[
                d:d+PATCH_SIZE[0],
                h:h+PATCH_SIZE[1],
                w:w+PATCH_SIZE[2]
            ]

            class_patch = class_volume[
                d:d+PATCH_SIZE[0],
                h:h+PATCH_SIZE[1],
                w:w+PATCH_SIZE[2]
            ]

            segmentation, _, _, reason = (
                make_targets_from_class_patch(class_patch)
            )

            if segmentation is None:
                raise RuntimeError(
                    f"Patch changed during extraction: {reason}"
                )

            images_mm[index, ..., 0] = ct_patch
            masks_mm[index] = segmentation
            classes_mm[index] = cls

            manifest_rows.append({
                "patient_id": patient_id,
                "split": split_name,
                "patch_index": index,
                "start_d": d,
                "start_h": h,
                "start_w": w,
                "classification": cls,
                "positive": is_positive
            })

        images_mm.flush()
        masks_mm.flush()
        classes_mm.flush()

        del images_mm, masks_mm, classes_mm
        images_mm = masks_mm = classes_mm = None

        manifest = pd.DataFrame(manifest_rows)

        manifest.to_csv(
            staging_dir / "patch_manifest.csv",
            index=False
        )

        # ----------------------------------------------------
        # 12. Commit the saved arrays
        # ----------------------------------------------------

        for filename in [
            "images.npy",
            "segmentation.npy",
            "classification.npy",
            "patch_manifest.csv"
        ]:

            os.replace(
                staging_dir / filename,
                output_dir / filename
            )

        status = {
            "status": "completed",
            "patient_id": patient_id,
            "split": split_name,
            "volume_shape": list(ct.shape),
            "total_candidate_patches": len(starts),
            "positive_patches": len(positive_records),
            "negative_patches": n_negative,
            "skipped_ambiguous": skipped_ambiguous,
            "skipped_mixed_class_ties": skipped_ties,
            "skipped_small": skipped_small,
            "saved_patches": n_patches,
            "image_shape": list(image_shape),
            "segmentation_shape": list(label_shape),
            "classification_shape": [n_patches],
            "created_at": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        }

        tmp_status = output_dir / "patient_patch_status.tmp.json"

        with open(tmp_status, "w", encoding="utf-8") as f:
            json.dump(status, f, indent=2)

        os.replace(tmp_status, status_path)

        print("\nPATCH EXTRACTION: PASS")
        print("Saved patches:", n_patches)
        print("Output:", output_dir)

        return status

    finally:

        del images_mm, masks_mm, classes_mm
        shutil.rmtree(staging_dir, ignore_errors=True)

        del ct, label, class_volume
        gc.collect()


print("Memory-efficient patient patch extractor defined.")


Memory-efficient patient patch extractor defined.


In [19]:

# ============================================================
# CELL #12B — FIX MISSING SHUTIL IMPORT
# ============================================================

import shutil
import gc
import os

print("shutil imported successfully.")

free_gb = shutil.disk_usage(PATCH_ROOT).free / (1024**3)

print(f"Available disk space: {free_gb:.2f} GiB")
print("Patch output directory:", PATCH_ROOT)


shutil imported successfully.
Available disk space: 19.50 GiB
Patch output directory: /kaggle/working/ribfrac-patch-dataset-final


In [21]:
# ============================================================
# CELL #12A — ONE-PATIENT PATCH EXTRACTION TEST
# ============================================================

TEST_PATIENT = "RibFrac128"

match = patient_split.loc[
    patient_split["public_id"].astype(str) == TEST_PATIENT
]

assert len(match) == 1
TEST_SPLIT = str(match.iloc[0]["split"])

print("Testing patient:", TEST_PATIENT)
print("Authoritative split:", TEST_SPLIT)

result = process_patient(
    TEST_PATIENT,
    TEST_SPLIT
)

print("\nONE-PATIENT EXTRACTION SUMMARY")
print("Patient:", result["patient_id"])
print("Positive patches:", result["positive_patches"])
print("Negative patches:", result["negative_patches"])
print("Saved patches:", result["saved_patches"])
print("Status:", result["status"])


Testing patient: RibFrac128
Authoritative split: train
RibFrac128: already extracted — SKIP

ONE-PATIENT EXTRACTION SUMMARY
Patient: RibFrac128
Positive patches: 4
Negative patches: 4
Saved patches: 8
Status: completed


In [22]:

# Check fracture annotation mappings for RibFrac128

patient_id = "RibFrac128"

mapping = patient_annotation_maps[patient_id]

print("Patient:", patient_id)
print("Annotation-to-class mapping:")

for annotation_id, class_code in sorted(mapping.items()):
    print(f"  Annotation {annotation_id}: Class {class_code}")

ambiguous_ids = [
    annotation_id
    for annotation_id, class_code in mapping.items()
    if class_code == AMBIGUOUS_LABEL_CODE
]

print("\nAmbiguous annotation IDs:", ambiguous_ids)


Patient: RibFrac128
Annotation-to-class mapping:
  Annotation 0: Class 0
  Annotation 1: Class -1
  Annotation 2: Class -1
  Annotation 3: Class -1
  Annotation 4: Class -1
  Annotation 5: Class -1
  Annotation 6: Class -1
  Annotation 7: Class 3
  Annotation 8: Class -1

Ambiguous annotation IDs: [1, 2, 3, 4, 5, 6, 8]


In [23]:

# ============================================================
# CELL #13 — RESUMABLE PATIENT-BY-PATIENT PATCH EXTRACTION
# RibFrac 49-patient pilot | Authoritative 500-patient split
# ============================================================

# ------------------------------------------------------------
# 1. Extraction configuration
# ------------------------------------------------------------

MAX_PATIENTS_PER_RUN = 5
RESUME_EXISTING = True
RETRY_FAILED = True

# ------------------------------------------------------------
# 2. Find patients with complete CT-label pairs
# ------------------------------------------------------------

image_ids = {
    p.name.removesuffix("-image.npy")
    for p in IMAGE_DIR.glob("*-image.npy")
}

label_ids = {
    p.name.removesuffix("-label.npy")
    for p in LABEL_DIR.glob("*-label.npy")
}

available_ids = image_ids & label_ids

assert image_ids == label_ids, (
    "CT and fracture-label inventories do not match."
)

assert len(available_ids) == 49, (
    f"Expected 49 available patients, "
    f"found {len(available_ids)}."
)

# Verify against Notebook 02 completed patients
assert available_ids == set(completed_ids), (
    "Saved patient pairs differ from "
    "Notebook 02 completed-status records."
)

# ------------------------------------------------------------
# 3. Keep authoritative patient assignments
# ------------------------------------------------------------

registry_to_process = patient_split.loc[
    patient_split["public_id"].astype(str).isin(available_ids)
].copy()

assert len(registry_to_process) == 49

split_priority = {
    "train": 0,
    "validation": 1,
    "test": 2
}

registry_to_process["_priority"] = (
    registry_to_process["split"].map(split_priority)
)

assert registry_to_process["_priority"].notna().all()

registry_to_process = (
    registry_to_process
    .sort_values(["_priority", "public_id"])
    .drop(columns="_priority")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Check previously saved patch outputs
# ------------------------------------------------------------

def saved_patch_output_valid(split_name, patient_id):

    output_dir = patient_patch_dir(
        split_name, patient_id
    )

    status_path = patient_status_path(
        split_name, patient_id
    )

    if not status_path.is_file():
        return False

    try:
        with open(status_path, "r", encoding="utf-8") as f:
            status = json.load(f)

        if status.get("status") != "completed":
            return False

        required = [
            output_dir / "images.npy",
            output_dir / "segmentation.npy",
            output_dir / "classification.npy",
            output_dir / "patch_manifest.csv"
        ]

        if not all(p.is_file() for p in required):
            return False

        images = np.load(
            required[0], mmap_mode="r", allow_pickle=False
        )

        masks = np.load(
            required[1], mmap_mode="r", allow_pickle=False
        )

        classes = np.load(
            required[2], mmap_mode="r", allow_pickle=False
        )

        manifest = pd.read_csv(required[3])

        n = len(images)

        valid = (
            n > 0
            and images.shape == (n, *PATCH_SIZE, 1)
            and masks.shape == (n, *PATCH_SIZE, 1)
            and classes.shape == (n,)
            and images.dtype == np.float32
            and masks.dtype == np.uint8
            and classes.dtype == np.int8
            and len(manifest) == n
            and n == int(status["saved_patches"])
        )

        del images, masks, classes

        return bool(valid)

    except (OSError, ValueError, KeyError, TypeError):
        return False


# ------------------------------------------------------------
# 5. Display available patient distribution
# ------------------------------------------------------------

print("=" * 70)
print("RIBFRAC — RESUMABLE PATCH EXTRACTION")
print("=" * 70)

print("Available patients:", len(registry_to_process))

print("\nPatients by authoritative split:")
print(registry_to_process["split"].value_counts())

print("\nMaximum patients attempted:", MAX_PATIENTS_PER_RUN)
print("Resume existing:", RESUME_EXISTING)
print("Retry failed:", RETRY_FAILED)

# ------------------------------------------------------------
# 6. Initialize counters
# ------------------------------------------------------------

attempted_this_run = 0
processed_this_run = 0
failed_this_run = 0
skipped_this_run = 0

run_results = []
failed_results = []

# ------------------------------------------------------------
# 7. Patient-by-patient extraction
# ------------------------------------------------------------

for _, row in registry_to_process.iterrows():

    patient_id = str(row["public_id"])
    split_name = str(row["split"])

    # Skip verified completed patches
    if RESUME_EXISTING and saved_patch_output_valid(
        split_name, patient_id
    ):

        skipped_this_run += 1
        print(f"{patient_id}: already completed — SKIP")
        continue

    # Respect retry preference
    status_path = patient_status_path(
        split_name, patient_id
    )

    if status_path.is_file() and not RETRY_FAILED:
        with open(status_path, "r", encoding="utf-8") as f:
            previous_status = json.load(f)

        if previous_status.get("status") == "failed":
            print(f"{patient_id}: previous failure — SKIP")
            continue

    # Limit attempts including failures
    if attempted_this_run >= MAX_PATIENTS_PER_RUN:
        print("\nBatch attempt limit reached.")
        break

    attempted_this_run += 1

    print("\n" + "=" * 70)
    print(
        f"PROCESSING {attempted_this_run}/"
        f"{MAX_PATIENTS_PER_RUN}"
    )
    print("=" * 70)

    print("Patient:", patient_id)
    print("Split:", split_name)

    try:

        # Extract and save patches using Cell #12
        result = process_patient(
            patient_id,
            split_name
        )

        # Confirm that extraction saved valid outputs
        if not saved_patch_output_valid(
            split_name, patient_id
        ):
            raise RuntimeError(
                "Saved patch output failed verification."
            )

        processed_this_run += 1
        run_results.append(result)

        print("\nSTATUS: COMPLETED")
        print("Saved patches:", result["saved_patches"])
        print("Positive patches:", result["positive_patches"])
        print("Negative patches:", result["negative_patches"])

    except Exception as exc:

        failed_this_run += 1
        error_text = repr(exc)

        print("\nSTATUS: FAILED")
        print("Error:", error_text)

        output_dir = patient_patch_dir(
            split_name, patient_id
        )

        output_dir.mkdir(
            parents=True, exist_ok=True
        )

        error_status = {
            "status": "failed",
            "patient_id": patient_id,
            "split": split_name,
            "error": error_text,
            "created_at": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        }

        with open(
            patient_status_path(split_name, patient_id),
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(error_status, f, indent=2)

        failed_results.append(error_status)

        # Stop on disk-space errors
        if (
            isinstance(exc, OSError)
            and getattr(exc, "errno", None) == 28
        ) or "insufficient disk space" in error_text.lower():
            print("Disk-space error. Stopping batch.")
            break

    finally:
        gc.collect()

# ------------------------------------------------------------
# 8. Display batch summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PATCH EXTRACTION BATCH SUMMARY")
print("=" * 70)

print("Available patients:", len(registry_to_process))
print("Patients attempted:", attempted_this_run)
print("Completed this run:", processed_this_run)
print("Failed this run:", failed_this_run)
print("Existing outputs skipped:", skipped_this_run)

# Count all verified patients in the current checkpoint
verified_ids = []

for _, row in registry_to_process.iterrows():

    if saved_patch_output_valid(
        str(row["split"]),
        str(row["public_id"])
    ):
        verified_ids.append(str(row["public_id"]))

print("\nTotal verified extracted patients:", len(verified_ids))
print("Available patients still without verified patches:",
      len(registry_to_process) - len(verified_ids))

print("\nPatch output directory:", PATCH_ROOT)

print("=" * 70)


RIBFRAC — RESUMABLE PATCH EXTRACTION
Available patients: 49

Patients by authoritative split:
split
train    36
test     13
Name: count, dtype: int64

Maximum patients attempted: 5
Resume existing: True
Retry failed: True

PROCESSING 1/5
Patient: RibFrac10
Split: train
PATIENT PATCH EXTRACTION
Patient: RibFrac10
Split: train
Volume shape: (381, 381, 451)
Candidate patches: 1300
Positive patches: 50
Selected negative patches: 50
Total patches to save: 100
Skipped ambiguous: 35
Skipped mixed-class ties: 0

PATCH EXTRACTION: PASS
Saved patches: 100
Output: /kaggle/working/ribfrac-patch-dataset-final/train/RibFrac10

STATUS: COMPLETED
Saved patches: 100
Positive patches: 50
Negative patches: 50

PROCESSING 2/5
Patient: RibFrac101
Split: train
PATIENT PATCH EXTRACTION
Patient: RibFrac101
Split: train
Volume shape: (350, 350, 441)
Candidate patches: 972
Positive patches: 32
Selected negative patches: 32
Total patches to save: 64
Skipped ambiguous: 12
Skipped mixed-class ties: 0

PATCH EXTRAC

In [24]:

# ============================================================
# CELL #14 — BUILD GLOBAL PATCH MANIFEST
# RibFrac 49-patient pilot
# ============================================================

manifest_frames = []

REQUIRED_COLUMNS = [
    "patient_id",
    "split",
    "patch_index",
    "start_d",
    "start_h",
    "start_w",
    "classification",
    "positive"
]

# ------------------------------------------------------------
# 1. Collect manifests for completed patients
# ------------------------------------------------------------

for split_name in ["train", "validation", "test"]:

    split_dir = PATCH_ROOT / split_name

    if not split_dir.exists():
        continue

    for patient_dir in sorted(split_dir.iterdir()):

        if not patient_dir.is_dir():
            continue

        patient_id = patient_dir.name

        status_path = (
            patient_dir / "patient_patch_status.json"
        )

        manifest_path = (
            patient_dir / "patch_manifest.csv"
        )

        if not status_path.is_file():
            continue

        with open(status_path, "r", encoding="utf-8") as f:
            status = json.load(f)

        # Include successfully completed patients only
        if status.get("status") != "completed":
            continue

        required_files = [
            patient_dir / "images.npy",
            patient_dir / "segmentation.npy",
            patient_dir / "classification.npy",
            manifest_path
        ]

        if not all(path.is_file() for path in required_files):
            raise FileNotFoundError(
                f"Incomplete patch outputs for {patient_id}"
            )

        patient_manifest = pd.read_csv(
            manifest_path,
            dtype={"patient_id": str, "split": str}
        )

        assert set(REQUIRED_COLUMNS).issubset(
            patient_manifest.columns
        ), f"Missing manifest columns: {patient_id}"

        assert (
            patient_manifest["patient_id"] == patient_id
        ).all(), f"Patient ID mismatch: {patient_id}"

        assert (
            patient_manifest["split"] == split_name
        ).all(), f"Split mismatch: {patient_id}"

        assert len(patient_manifest) == int(
            status["saved_patches"]
        ), f"Patch count mismatch: {patient_id}"

        assert not patient_manifest[
            "patch_index"
        ].duplicated().any()

        manifest_frames.append(
            patient_manifest[REQUIRED_COLUMNS]
        )

# ------------------------------------------------------------
# 2. Combine individual patient manifests
# ------------------------------------------------------------

if manifest_frames:

    global_manifest = pd.concat(
        manifest_frames,
        ignore_index=True
    )

else:

    global_manifest = pd.DataFrame(
        columns=REQUIRED_COLUMNS
    )

# ------------------------------------------------------------
# 3. Verify global manifest consistency
# ------------------------------------------------------------

if len(global_manifest):

    assert not global_manifest.duplicated(
        subset=["patient_id", "split", "patch_index"]
    ).any(), "Duplicate patch records detected."

    assert global_manifest[
        "classification"
    ].isin([0, 1, 2, 3, 4]).all()

    assert global_manifest[
        "positive"
    ].isin([0, 1]).all()

    assert (
        (global_manifest["classification"] != 0).astype(int)
        == global_manifest["positive"]
    ).all(), "Classification/positive-label mismatch."

# ------------------------------------------------------------
# 4. Save global manifest
# ------------------------------------------------------------

PATCH_METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GLOBAL_MANIFEST_PATH = (
    PATCH_METADATA_DIR / "patch_manifest.csv"
)

tmp_path = (
    PATCH_METADATA_DIR / "patch_manifest.tmp.csv"
)

global_manifest.to_csv(
    tmp_path,
    index=False
)

os.replace(tmp_path, GLOBAL_MANIFEST_PATH)

# ------------------------------------------------------------
# 5. Display extraction summary
# ------------------------------------------------------------

print("=" * 70)
print("GLOBAL PATCH MANIFEST")
print("=" * 70)

print("Manifest saved:", GLOBAL_MANIFEST_PATH)
print("Total saved patches:", len(global_manifest))

if len(global_manifest):

    print(
        "Patients represented:",
        global_manifest["patient_id"].nunique()
    )

    print("\nPatch counts by split:")
    print(global_manifest["split"].value_counts())

    print("\nPatch counts by classification:")
    print(
        global_manifest["classification"]
        .value_counts()
        .sort_index()
    )

    print("\nPositive and negative patch counts:")
    print(global_manifest["positive"].value_counts())

    print("\nGlobal manifest validation: PASS")

else:
    print("No completed patient patches found yet.")

print("=" * 70)


GLOBAL PATCH MANIFEST
Manifest saved: /kaggle/working/ribfrac-patch-dataset-final/metadata/patch_manifest.csv
Total saved patches: 302
Patients represented: 5

Patch counts by split:
split
train    302
Name: count, dtype: int64

Patch counts by classification:
classification
0    151
1     30
2     69
3     40
4     12
Name: count, dtype: int64

Positive and negative patch counts:
positive
1    151
0    151
Name: count, dtype: int64

Global manifest validation: PASS


In [26]:
# Cell 15 — Patient patch summary and class distribution

summary_rows = []

for split_name in ["train", "validation", "test"]:

    split_dir = PATCH_ROOT / split_name

    if not split_dir.exists():
        continue

    for patient_dir in sorted(split_dir.iterdir()):

        if not patient_dir.is_dir():
            continue

        status_path = patient_dir / "patient_patch_status.json"

        if not status_path.exists():
            continue

        with open(status_path, "r", encoding="utf-8") as f:
            summary_rows.append(json.load(f))

patient_summary = pd.DataFrame(summary_rows)

PATIENT_SUMMARY_PATH = (
    PATCH_METADATA_DIR / "patient_patch_summary.csv"
)

patient_summary.to_csv(
    PATIENT_SUMMARY_PATH,
    index=False
)

if len(global_manifest):
    class_distribution = (
        global_manifest
        .groupby(["split", "classification"])
        .size()
        .reset_index(name="patch_count")
    )
else:
    class_distribution = pd.DataFrame(
        columns=["split", "classification", "patch_count"]
    )

CLASS_DISTRIBUTION_PATH = (
    PATCH_METADATA_DIR / "class_distribution.csv"
)

class_distribution.to_csv(
    CLASS_DISTRIBUTION_PATH,
    index=False
)

print("Patient summary:", PATIENT_SUMMARY_PATH)
print("Class distribution:", CLASS_DISTRIBUTION_PATH)
print("\nClass distribution:")
print(class_distribution)

Patient summary: /kaggle/working/ribfrac-patch-dataset-final/metadata/patient_patch_summary.csv
Class distribution: /kaggle/working/ribfrac-patch-dataset-final/metadata/class_distribution.csv

Class distribution:
   split  classification  patch_count
0  train               0          151
1  train               1           30
2  train               2           69
3  train               3           40
4  train               4           12


In [27]:

# Cell 15A — Verify patch summary and class distribution

print("=" * 65)
print("RIBFRAC PATCH EXTRACTION SUMMARY")
print("=" * 65)

print("\nPatients with recorded extraction status:")
print(len(patient_summary))

if not patient_summary.empty:

    print("\nPatient processing status:")
    print(patient_summary["status"].value_counts())

    print("\nCompleted patients by split:")

    completed_summary = patient_summary[
        patient_summary["status"] == "completed"
    ]

    print(completed_summary["split"].value_counts())

if not global_manifest.empty:

    print("\nTotal saved patches:", len(global_manifest))

    print("\nPatch distribution by split:")
    print(global_manifest["split"].value_counts())

    print("\nClassification distribution:")

    class_summary = (
        global_manifest["classification"]
        .value_counts()
        .sort_index()
        .rename(index=CLASS_NAMES)
    )

    print(class_summary)

print("\nPATIENT PATCH SUMMARY: GENERATED")
print("=" * 65)


RIBFRAC PATCH EXTRACTION SUMMARY

Patients with recorded extraction status:
6

Patient processing status:
status
completed    5
failed       1
Name: count, dtype: int64

Completed patients by split:
split
train    5
Name: count, dtype: int64

Total saved patches: 302

Patch distribution by split:
split
train    302
Name: count, dtype: int64

Classification distribution:
classification
background       151
displaced         30
non-displaced     69
buckle            40
segmental         12
Name: count, dtype: int64

PATIENT PATCH SUMMARY: GENERATED


In [28]:
# Cell 16 — Save extraction configuration

config = {
    "notebook": "03_Patch_Extraction_RibFrac_Final_Persistent_500Patients",
    "seed": SEED,
    "patch_size": list(PATCH_SIZE),
    "stride": list(STRIDE),
    "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
    "minimum_fracture_voxels": MIN_FRACTURE_VOXELS,
    "ambiguous_label_code": AMBIGUOUS_LABEL_CODE,
    "classification_classes": CLASS_NAMES,
    "label_code_to_class": LABEL_CODE_TO_CLASS,
    "patient_counts": EXPECTED_SPLIT_COUNTS,
    "patient_level_split": True,
    "patch_level_split": False,
    "official_ribfrac_test_ground_truth_used": False,
    "internal_test_is_labelled_holdout": True,
    "input_dataset": str(DATASET_ROOT),
}

CONFIG_PATH = (
    PATCH_METADATA_DIR / "patch_extraction_config.json"
)

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Configuration saved:", CONFIG_PATH)

Configuration saved: /kaggle/working/ribfrac-patch-dataset-final/metadata/patch_extraction_config.json


In [29]:
assert CONFIG_PATH.is_file()

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    saved_config = json.load(f)

assert saved_config["patch_size"] == [64, 64, 64]
assert saved_config["stride"] == [32, 32, 32]
assert saved_config["seed"] == 42

print("Patch extraction configuration: PASS")

Patch extraction configuration: PASS


In [30]:
# Cell 17 — Validate saved patient patch files

validation_errors = []

for _, row in patient_summary.iterrows():

    if row.get("status") != "completed":
        continue

    patient_id = str(row["patient_id"])
    split_name = str(row["split"])
    patient_dir = patient_patch_dir(split_name, patient_id)

    image_path = patient_dir / "images.npy"
    segmentation_path = patient_dir / "segmentation.npy"
    classification_path = patient_dir / "classification.npy"
    manifest_path = patient_dir / "patch_manifest.csv"

    required = [
        image_path,
        segmentation_path,
        classification_path,
        manifest_path,
    ]

    for path in required:
        if not path.exists():
            validation_errors.append(f"Missing: {path}")

    if not all(path.exists() for path in required):
        continue

    images = np.load(image_path, mmap_mode="r")
    segmentations = np.load(segmentation_path, mmap_mode="r")
    classifications = np.load(classification_path, mmap_mode="r")
    manifest = pd.read_csv(manifest_path)

    if images.ndim != 5:
        validation_errors.append(
            f"{patient_id}: images ndim={images.ndim}"
        )

    if images.shape[1:] != (64, 64, 64, 1):
        validation_errors.append(
            f"{patient_id}: bad image shape {images.shape}"
        )

    if segmentations.shape[1:] != (64, 64, 64, 1):
        validation_errors.append(
            f"{patient_id}: bad segmentation shape {segmentations.shape}"
        )

    if len(images) != len(segmentations):
        validation_errors.append(
            f"{patient_id}: image/segmentation count mismatch"
        )

    if len(images) != len(classifications):
        validation_errors.append(
            f"{patient_id}: image/classification count mismatch"
        )

    if len(images) != len(manifest):
        validation_errors.append(
            f"{patient_id}: manifest count mismatch"
        )

    if not set(np.unique(segmentations).tolist()).issubset({0, 1}):
        validation_errors.append(
            f"{patient_id}: invalid segmentation values"
        )

    if not set(np.unique(classifications).tolist()).issubset(
        {0, 1, 2, 3, 4}
    ):
        validation_errors.append(
            f"{patient_id}: invalid classification values"
        )

    del images, segmentations, classifications
    gc.collect()

if validation_errors:
    print("VALIDATION FAILED")
    for error in validation_errors[:50]:
        print(error)
    raise AssertionError(
        f"{len(validation_errors)} validation errors found."
    )

print("Saved patch-file validation: PASS")

Saved patch-file validation: PASS


In [31]:

# Cell 17A — Summarize validated patient patches

completed_summary = patient_summary.loc[
    patient_summary["status"] == "completed"
].copy()

print("=" * 60)
print("RIBFRAC PATCH VALIDATION SUMMARY")
print("=" * 60)

print("Available preprocessed patients:", 49)
print("Patients with completed extraction:", len(completed_summary))

if len(completed_summary):

    total_patches = pd.to_numeric(
        completed_summary["saved_patches"],
        errors="raise"
    ).sum()

    print("Total saved patches:", int(total_patches))

    print("\nCompleted patients by split:")
    print(completed_summary["split"].value_counts())

    print("\nClassification distribution:")
    print(
        global_manifest["classification"]
        .value_counts()
        .sort_index()
        .rename(index=CLASS_NAMES)
    )

    assert int(total_patches) == len(global_manifest), (
        "Patch summary and global manifest counts differ."
    )

    print("\nPATCH DATASET SUMMARY: PASS")

else:
    print("No completed patient patches found.")

print("=" * 60)


RIBFRAC PATCH VALIDATION SUMMARY
Available preprocessed patients: 49
Patients with completed extraction: 5
Total saved patches: 302

Completed patients by split:
split
train    5
Name: count, dtype: int64

Classification distribution:
classification
background       151
displaced         30
non-displaced     69
buckle            40
segmental         12
Name: count, dtype: int64

PATCH DATASET SUMMARY: PASS


In [32]:
# Cell 18 — Final Notebook 03 report

print("=" * 80)
print("NOTEBOOK 03 — PERSISTENT PATCH EXTRACTION REPORT")
print("=" * 80)

completed_patients = (
    patient_summary.loc[
        patient_summary["status"] == "completed"
    ]
    if len(patient_summary)
    else pd.DataFrame()
)

print("\nCompleted patients:", len(completed_patients))

if len(patient_summary):
    print("\nPatient status:")
    print(patient_summary["status"].value_counts())

if len(global_manifest):
    print("\nPatch counts by split:")
    print(global_manifest["split"].value_counts())

    print("\nClassification distribution:")
    print(
        global_manifest["classification"]
        .value_counts()
        .sort_index()
    )

print("\nPatch output:", PATCH_ROOT)

total_size_gb = (
    sum(
        p.stat().st_size
        for p in PATCH_ROOT.rglob("*")
        if p.is_file()
    )
    / (1024 ** 3)
)

print("Patch dataset size:", f"{total_size_gb:.3f} GB")

NOTEBOOK 03 — PERSISTENT PATCH EXTRACTION REPORT

Completed patients: 5

Patient status:
status
completed    5
failed       1
Name: count, dtype: int64

Patch counts by split:
split
train    302
Name: count, dtype: int64

Classification distribution:
classification
0    151
1     30
2     69
3     40
4     12
Name: count, dtype: int64

Patch output: /kaggle/working/ribfrac-patch-dataset-final
Patch dataset size: 0.369 GB


In [33]:

# Cell 18A — Pilot patch extraction progress

completed_count = len(completed_patients)
available_count = 49

print("=" * 60)
print("RIBFRAC 49-PATIENT PATCH EXTRACTION PROGRESS")
print("=" * 60)

print("Available patients:", available_count)
print("Completed patch extraction:", completed_count)
print("Remaining:", available_count - completed_count)

print(
    "Completion percentage:",
    f"{100 * completed_count / available_count:.2f}%"
)

print("Total extracted patches:", len(global_manifest))

assert completed_count <= available_count

if completed_count == available_count:
    print("\n49-PATIENT PATCH EXTRACTION: COMPLETE")
else:
    print("\n49-PATIENT PATCH EXTRACTION: IN PROGRESS")


RIBFRAC 49-PATIENT PATCH EXTRACTION PROGRESS
Available patients: 49
Completed patch extraction: 5
Remaining: 44
Completion percentage: 10.20%
Total extracted patches: 302

49-PATIENT PATCH EXTRACTION: IN PROGRESS


# Notebook 03 completion criteria

Notebook 03 is ready for Notebook 04/05 when:

1. The persistent Notebook 02 dataset is successfully mounted.
2. The 500-patient registry is verified.
3. The 350/75/75 patient split is verified.
4. Annotation IDs are mapped using RibFrac metadata.
5. Ambiguous annotations (`label_code = -1`) are excluded.
6. CT patches have shape `(64,64,64,1)`.
7. Segmentation targets contain only `{0,1}`.
8. Classification targets contain only `{0,1,2,3,4}`.
9. Positive patches contain valid fracture voxels.
10. Negative patches contain no valid fracture voxels.
11. Mixed-class ties are excluded.
12. CT, segmentation, and classification patch counts agree.
13. Patch manifests can be reloaded.
14. Patient-level split integrity is preserved.
15. No patch-level random splitting is performed.
16. Extraction is performed patient-by-patient and can resume.

## First execution

Keep:

```python
MAX_PATIENTS_PER_RUN = 1
RESUME_EXISTING = True
```

Run the notebook through Cell 18 and verify the first patient before increasing the batch size.

In [34]:
from pathlib import Path
import pandas as pd

PATCH_ROOT = Path("/kaggle/working/ribfrac-patch-dataset-final")

print("Patch root exists:", PATCH_ROOT.exists())

for split in ["train", "validation", "test"]:
    split_dir = PATCH_ROOT / split
    print(f"\n{split.upper()} directory exists:", split_dir.exists())

    if split_dir.exists():
        patients = [
            p for p in split_dir.iterdir()
            if p.is_dir()
        ]
        print("Patient directories:", len(patients))
        print("Examples:", [p.name for p in patients[:5]])

manifest_path = PATCH_ROOT / "metadata" / "patch_manifest.csv"
print("\nManifest exists:", manifest_path.exists())

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print("Manifest rows:", len(manifest))
    print("Manifest columns:", manifest.columns.tolist())
    if "split" in manifest.columns:
        print("\nPatches by split:")
        print(manifest["split"].value_counts())

Patch root exists: True

TRAIN directory exists: True
Patient directories: 6
Examples: ['RibFrac128', 'RibFrac104', 'RibFrac10', 'RibFrac103', 'RibFrac102']

VALIDATION directory exists: False

TEST directory exists: False

Manifest exists: True
Manifest rows: 302
Manifest columns: ['patient_id', 'split', 'patch_index', 'start_d', 'start_h', 'start_w', 'classification', 'positive']

Patches by split:
split
train    302
Name: count, dtype: int64


In [36]:
from pathlib import Path
import pandas as pd

DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

METADATA_ROOT = DATASET_ROOT / "metadata"

print("Metadata directory exists:", METADATA_ROOT.exists())

if METADATA_ROOT.exists():
    for file in METADATA_ROOT.iterdir():
        print(file.name)

    split_files = list(METADATA_ROOT.glob("*split*.csv"))

    for file in split_files:
        print(f"\n--- {file.name} ---")

        df = pd.read_csv(file)

        print("Columns:", df.columns.tolist())
        print(df.head())

        for col in df.columns:
            if "split" in col.lower():
                print("\nPatient counts by split:")
                print(df[col].value_counts(dropna=False))

Metadata directory exists: True
combined_ribfrac_metadata.csv
patient_source_registry_500.csv
patient_split_500.csv
persisted_patient_inventory.csv
preprocessing_config_500.json
processing_status.csv
processing_status_before_merge.csv

--- patient_split_500.csv ---
Columns: ['public_id', 'split']
    public_id       split
0    RibFrac1  validation
1   RibFrac10       train
2  RibFrac100  validation
3  RibFrac101       train
4  RibFrac102       train

Patient counts by split:
split
train         350
validation     75
test           75
Name: count, dtype: int64


In [37]:
from pathlib import Path
import pandas as pd

PATCH_ROOT = Path("/kaggle/working/ribfrac-patch-dataset-final")
METADATA_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final/metadata"
)

split_df = pd.read_csv(METADATA_ROOT / "patient_split_500.csv")
manifest_df = pd.read_csv(PATCH_ROOT / "metadata" / "patch_manifest.csv")

# Standardise patient identifiers
split_df["public_id"] = split_df["public_id"].astype(str).str.strip()
manifest_df["patient_id"] = manifest_df["patient_id"].astype(str).str.strip()

# Count processed patients and patches
print("PATIENT SPLIT REGISTRY")
print(split_df["split"].value_counts())

print("\nPROCESSED PATIENTS BY SPLIT")
processed = manifest_df[["patient_id", "split"]].drop_duplicates()
print(processed.groupby("split")["patient_id"].nunique())

print("\nPATCHES BY SPLIT")
print(manifest_df["split"].value_counts())

# Find patients assigned to train/validation but not yet processed
completed_ids = set(processed["patient_id"])
split_df["processed"] = split_df["public_id"].isin(completed_ids)

print("\nUNPROCESSED PATIENTS BY SPLIT")
print(
    split_df.loc[~split_df["processed"], "split"]
    .value_counts()
)

print("\nEXAMPLES OF UNPROCESSED TRAIN PATIENTS")
print(
    split_df.loc[
        (~split_df["processed"]) & (split_df["split"] == "train"),
        "public_id"
    ].head(10).tolist()
)

print("\nEXAMPLES OF UNPROCESSED VALIDATION PATIENTS")
print(
    split_df.loc[
        (~split_df["processed"]) & (split_df["split"] == "validation"),
        "public_id"
    ].head(10).tolist()
)

PATIENT SPLIT REGISTRY
split
train         350
validation     75
test           75
Name: count, dtype: int64

PROCESSED PATIENTS BY SPLIT
split
train    5
Name: patient_id, dtype: int64

PATCHES BY SPLIT
split
train    302
Name: count, dtype: int64

UNPROCESSED PATIENTS BY SPLIT
split
train         345
validation     75
test           75
Name: count, dtype: int64

EXAMPLES OF UNPROCESSED TRAIN PATIENTS
['RibFrac104', 'RibFrac105', 'RibFrac106', 'RibFrac108', 'RibFrac11', 'RibFrac110', 'RibFrac111', 'RibFrac113', 'RibFrac117', 'RibFrac118']

EXAMPLES OF UNPROCESSED VALIDATION PATIENTS
['RibFrac1', 'RibFrac100', 'RibFrac112', 'RibFrac115', 'RibFrac116', 'RibFrac119', 'RibFrac126', 'RibFrac127', 'RibFrac137', 'RibFrac150']


In [38]:

from pathlib import Path
import pandas as pd

DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

METADATA_ROOT = DATASET_ROOT / "metadata"
IMAGE_ROOT = DATASET_ROOT / "images"
LABEL_ROOT = DATASET_ROOT / "labels"

split_df = pd.read_csv(METADATA_ROOT / "patient_split_500.csv")
status_df = pd.read_csv(METADATA_ROOT / "processing_status.csv")

print("Processing-status columns:")
print(status_df.columns.tolist())

print("\nProcessing-status preview:")
print(status_df.head())

print("\nMounted image files:", len(list(IMAGE_ROOT.glob("*"))))
print("Mounted label files:", len(list(LABEL_ROOT.glob("*"))))

print("\nImage examples:")
print([p.name for p in list(IMAGE_ROOT.iterdir())[:10]])

print("\nLabel examples:")
print([p.name for p in list(LABEL_ROOT.iterdir())[:10]])


Processing-status columns:
['public_id', 'source', 'split', 'status', 'ct_path', 'label_path', 'ct_shape', 'label_shape', 'ct_dtype', 'label_dtype', 'original_spacing', 'target_spacing', 'ct_min', 'ct_max', 'label_ids', 'ct_size_mb', 'label_size_mb', 'started_at', 'completed_at', 'error']

Processing-status preview:
    public_id       source       split     status  \
0    RibFrac1  train_part1  validation    pending   
1   RibFrac10  train_part1       train  completed   
2  RibFrac100  train_part1  validation    pending   
3  RibFrac101  train_part1       train  completed   
4  RibFrac102  train_part1       train  completed   

                       ct_path                   label_path         ct_shape  \
0                          NaN                          NaN              NaN   
1   images/RibFrac10-image.npy   labels/RibFrac10-label.npy  (381, 381, 451)   
2                          NaN                          NaN              NaN   
3  images/RibFrac101-image.npy  labels/RibF

In [39]:

from pathlib import Path
import pandas as pd

DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-preprocessed-final"
)

IMAGE_ROOT = DATASET_ROOT / "images"
LABEL_ROOT = DATASET_ROOT / "labels"
STATUS_PATH = DATASET_ROOT / "metadata" / "processing_status.csv"
SPLIT_PATH = DATASET_ROOT / "metadata" / "patient_split_500.csv"

status = pd.read_csv(STATUS_PATH)
splits = pd.read_csv(SPLIT_PATH)

available = []

for image_path in sorted(IMAGE_ROOT.glob("*-image.npy")):
    patient_id = image_path.name.replace("-image.npy", "")
    label_path = LABEL_ROOT / f"{patient_id}-label.npy"

    status_row = status.loc[status["public_id"] == patient_id]
    split_row = splits.loc[splits["public_id"] == patient_id]

    available.append({
        "patient_id": patient_id,
        "image_exists": image_path.exists(),
        "label_exists": label_path.exists(),
        "registry_status": (
            status_row["status"].iloc[0]
            if not status_row.empty else "not found"
        ),
        "assigned_split": (
            split_row["split"].iloc[0]
            if not split_row.empty else "not found"
        )
    })

available_df = pd.DataFrame(available)

print("Available patient files:", len(available_df))
print(available_df.to_string(index=False))

print("\nAvailable patients by assigned split:")
print(available_df.groupby("assigned_split").size())


Available patient files: 49
patient_id  image_exists  label_exists registry_status assigned_split
 RibFrac10          True          True       completed          train
RibFrac101          True          True       completed          train
RibFrac102          True          True       completed          train
RibFrac103          True          True       completed          train
RibFrac104          True          True       completed          train
RibFrac105          True          True       completed          train
RibFrac106          True          True       completed          train
RibFrac107          True          True       completed           test
RibFrac108          True          True       completed          train
RibFrac109          True          True       completed           test
 RibFrac11          True          True       completed          train
RibFrac110          True          True       completed          train
RibFrac111          True          True       completed        

In [40]:
import kagglehub

kagglehub.dataset_upload(
    "waboke/ribfrac-patches-pilot",
    str(PATCH_ROOT),
    version_notes="Initial RibFrac patch extraction pilot"
)

Uploading Dataset https://kaggle.com/datasets/waboke/ribfrac-patches-pilot ...
Starting upload for file /kaggle/working/ribfrac-patch-dataset-final/train/RibFrac128/patient_patch_status.json

Uploading: 100%|██████████| 524/524 [00:00<00:00, 2.02kB/s]

Upload successful: /kaggle/working/ribfrac-patch-dataset-final/train/RibFrac128/patient_patch_status.json (524B)
Starting upload for file /kaggle/working/ribfrac-patch-dataset-final/train/RibFrac128/segmentation.npy



Uploading: 100%|██████████| 2.10M/2.10M [00:00<00:00, 8.91MB/s]

Upload successful: /kaggle/working/ribfrac-patch-dataset-final/train/RibFrac128/segmentation.npy (2MB)
Starting upload for file /kaggle/working/ribfrac-patch-dataset-final/train/RibFrac128/patch_manifest.csv



Uploading: 100%|██████████| 348/348 [00:00<00:00, 1.67kB/s]

... (52 outputs left out)

Upload successful: /kaggle/working/ribfrac-patch-dataset-final/metadata/class_distribution.csv (89B)
Starting upload for file /kaggle/working/ribfrac-patch-dataset-final/metadata/patient_patch_summary.csv



Uploading: 100%|██████████| 1.09k/1.09k [00:00<00:00, 4.96kB/s]

Upload successful: /kaggle/working/ribfrac-patch-dataset-final/metadata/patient_patch_summary.csv (1KB)


Your dataset has been created.
Files are being processed...
See at: https://kaggle.com/datasets/waboke/ribfrac-patches-pilot
